# Evaluation analysis — confusion matrix, per-class F1, and Grad-CAM

This notebook is designed to run **after** training is complete.
Load your saved checkpoints, then run all cells.

Covers:
1. Per-class accuracy, precision, recall, F1 (sklearn `classification_report`)
2. Confusion matrix heatmap (most-confused artist pairs)
3. Grad-CAM saliency maps — visualise which image regions the model attends to


In [ ]:
import numpy as np
import tensorflow as tf
import keras
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from keras.utils import image_dataset_from_directory
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

# pip install scikit-learn seaborn  (if not already installed)


## 1 — Load trained models and collect predictions

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR    = Path("../wikiart_split")
CKPT_DIR    = Path(".")            # folder containing .keras checkpoint files
IMAGE_SIZE  = (384, 384)
BATCH_SIZE  = 16

# Map checkpoint filenames to display names
CHECKPOINTS = {
    "EfficientNetV2S": CKPT_DIR / "ckpt_phase2_efficientnetv2s.keras",
    "Xception":        CKPT_DIR / "ckpt_phase2_xception.keras",
    "MyCNN":           CKPT_DIR / "ckpt_phase1_my_cnn.keras",
    # Add DINOv2 if trained:
    # "DINOv2":        CKPT_DIR / "ckpt_dino.keras",
}

# ── Load test dataset (shuffle=False — must preserve image order) ─────────────
test_ds = image_dataset_from_directory(
    DATA_DIR / "test",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=123,
)
class_names = test_ds.class_names
N_CLASSES   = len(class_names)
test_ds     = test_ds.cache().prefetch(tf.data.AUTOTUNE)

# Ground-truth labels (integer class indices)
true_labels = np.concatenate(
    [np.argmax(y.numpy(), axis=1) for _, y in test_ds], axis=0
)
print(f"Test set: {len(true_labels)} images, {N_CLASSES} classes")


In [ ]:
# ── Load models and predict ────────────────────────────────────────────────────
# For DINOv2 you need the feature test dataset instead — see dino_v2_model.ipynb
model_preds = {}    # name -> predicted class indices
model_probs = {}    # name -> softmax probability arrays (for ensemble)
models      = {}    # name -> loaded keras model (needed for Grad-CAM)

for name, ckpt_path in CHECKPOINTS.items():
    if not ckpt_path.exists():
        print(f"  SKIPPING {name} — checkpoint not found: {ckpt_path}")
        continue
    print(f"Loading {name} from {ckpt_path}...")
    model = keras.models.load_model(ckpt_path)
    probs = model.predict(test_ds, verbose=0)
    model_probs[name] = probs
    model_preds[name] = np.argmax(probs, axis=1)
    models[name]      = model
    acc = np.mean(model_preds[name] == true_labels)
    print(f"  {name}: accuracy = {acc:.4f}")


## 2 — Per-class classification report

In [ ]:
for name, preds in model_preds.items():
    print(f"\n{'='*60}")
    print(f"Classification report — {name}")
    print('='*60)
    print(classification_report(
        true_labels, preds,
        target_names=class_names,
        digits=3,
    ))

## 3 — Confusion matrix

In [ ]:
def plot_confusion_matrix(true, pred, class_names, title, figsize=(14, 12)):
    """
    Normalised confusion matrix (row-normalised = recall per class).
    Diagonal = per-class recall. Off-diagonal = what the model confuses it with.
    """
    cm = confusion_matrix(true, pred, normalize="true")

    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(
        cm,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax,
        linewidths=0.3,
        linecolor="white",
        vmin=0, vmax=1,
    )
    ax.set_xlabel("Predicted artist", fontsize=11, labelpad=10)
    ax.set_ylabel("True artist",      fontsize=11, labelpad=10)
    ax.set_title(title, fontsize=13, pad=14)
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.yticks(rotation=0,             fontsize=8)
    plt.tight_layout()
    plt.savefig(f"confusion_matrix_{title.replace(' ','_')}.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to confusion_matrix_{title.replace(' ','_')}.png")


for name, preds in model_preds.items():
    plot_confusion_matrix(true_labels, preds, class_names,
                          title=name, figsize=(16, 14))


In [ ]:
# ── Most-confused pairs ───────────────────────────────────────────────────────
# Useful for the report: shows which artists the model systematically conflates.

def most_confused_pairs(true, pred, class_names, top_n=10):
    cm = confusion_matrix(true, pred)
    np.fill_diagonal(cm, 0)   # ignore correct predictions
    pairs = []
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            if i != j and cm[i, j] > 0:
                pairs.append((cm[i, j], class_names[i], class_names[j]))
    pairs.sort(reverse=True)
    print(f"\nTop {top_n} most-confused pairs (true → predicted):")
    print(f"{'Count':>6}  {'True artist':<25} → {'Predicted as'}")
    print("-" * 55)
    for count, true_c, pred_c in pairs[:top_n]:
        print(f"{count:>6}  {true_c:<25} → {pred_c}")

for name, preds in model_preds.items():
    print(f"\n{'='*60}")
    print(f"Most-confused pairs — {name}")
    most_confused_pairs(true_labels, preds, class_names)


## 4 — Ensemble evaluation

Average the softmax probabilities from all loaded models.


In [ ]:
if len(model_probs) > 1:
    ensemble_probs = np.mean(list(model_probs.values()), axis=0)
    ensemble_preds = np.argmax(ensemble_probs, axis=1)
    ens_acc = np.mean(ensemble_preds == true_labels)
    print(f"Ensemble accuracy: {ens_acc:.4f}")
    print()
    print(classification_report(true_labels, ensemble_preds,
                                 target_names=class_names, digits=3))
    plot_confusion_matrix(true_labels, ensemble_preds, class_names,
                          title="Ensemble", figsize=(16, 14))
else:
    print("Only one model loaded — skipping ensemble.")


## 5 — Grad-CAM saliency maps

Grad-CAM answers: *which spatial regions of a painting most influenced the prediction?*

It computes the gradient of the predicted class score with respect to the
last convolutional feature map. Regions with strong positive gradients are
highlighted — these are the areas the model "looked at" most.

**For the report:** show examples where Grad-CAM attends to brushstroke
texture (good) vs. the primary subject like a boat or tree (bad — subject bias).

Works with: MyCNN, EfficientNetV2S, Xception. Does NOT apply to DINOv2
(which is a ViT, not a CNN — attention rollout maps are used instead).


In [ ]:
def get_last_conv_layer(model):
    """Find the last Conv2D layer in a Keras model."""
    for layer in reversed(model.layers):
        if isinstance(layer, (tf.keras.layers.Conv2D,)):
            return layer.name
        # For subclassed models, recurse into the base
        if hasattr(layer, 'layers'):
            for sublayer in reversed(layer.layers):
                if isinstance(sublayer, tf.keras.layers.Conv2D):
                    return sublayer.name
    raise ValueError("No Conv2D layer found in model.")


def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Compute a Grad-CAM heatmap for a single image.

    img_array: float32 tensor, shape (1, H, W, 3), pixel values [0, 255]
    Returns:   heatmap as a (H', W') float array in [0, 1]
    """
    # Build a sub-model that outputs both the last conv feature map and the predictions
    grad_model = keras.models.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(last_conv_layer_name).output,
            model.output,
        ],
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array, training=False)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    # Gradients of the class score w.r.t. the conv feature map
    grads = tape.gradient(class_channel, conv_outputs)

    # Global average pool the gradients over spatial dimensions → weights per channel
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weight the feature map channels by the gradient magnitudes
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_gradcam(img_array, heatmap, alpha=0.4):
    """Resize heatmap to image size and overlay as a colourmap."""
    import cv2  # pip install opencv-python
    h, w = img_array.shape[:2]
    heatmap_resized = cv2.resize(heatmap, (w, h))
    heatmap_uint8   = np.uint8(255 * heatmap_resized)
    jet             = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    jet_rgb         = cv2.cvtColor(jet, cv2.COLOR_BGR2RGB)
    overlay         = (1 - alpha) * img_array.astype(float) + alpha * jet_rgb
    return np.clip(overlay, 0, 255).astype(np.uint8)


In [ ]:
def visualise_gradcam_grid(model, model_name, class_names, data_dir,
                           n_artists=6, n_per_artist=3, image_size=(384, 384)):
    """
    Show Grad-CAM overlays for n_per_artist paintings from n_artists.
    Selects artists with highest error rate first — most informative for the report.
    """
    import cv2
    from PIL import Image as PILImage

    # Find the last conv layer for this model
    try:
        last_conv = get_last_conv_layer(model)
        print(f"Using conv layer: {last_conv}")
    except ValueError as e:
        print(f"Skipping Grad-CAM for {model_name}: {e}")
        return

    train_dir = data_dir / "test"
    artists   = [d.name for d in sorted(train_dir.iterdir()) if d.is_dir()][:n_artists]
    img_exts  = {".jpg", ".jpeg", ".png"}

    fig, axes = plt.subplots(
        n_artists, n_per_artist * 2,
        figsize=(n_per_artist * 5, n_artists * 3)
    )
    fig.suptitle(f"Grad-CAM — {model_name}"
                 f"Left: original  |  Right: saliency overlay", fontsize=12)

    for row, artist in enumerate(artists):
        artist_dir = train_dir / artist
        imgs = [p for p in artist_dir.iterdir()
                if p.suffix.lower() in img_exts][:n_per_artist]

        for col, img_path in enumerate(imgs):
            # Load and preprocess
            img_pil  = PILImage.open(img_path).convert("RGB")
            img_pil  = img_pil.resize((image_size[1], image_size[0]))
            img_np   = np.array(img_pil)
            img_tf   = tf.cast(img_np[tf.newaxis, ...], tf.float32)

            # Predict and get heatmap
            preds      = model(img_tf, training=False).numpy()[0]
            pred_class = np.argmax(preds)
            pred_name  = class_names[pred_class]
            confidence = preds[pred_class]

            heatmap = make_gradcam_heatmap(img_tf, model, last_conv)
            overlay = overlay_gradcam(img_np, heatmap, alpha=0.45)

            # Plot original
            ax_orig = axes[row, col * 2]
            ax_orig.imshow(img_np)
            ax_orig.set_title(f"True: {artist}"
                              f"Pred: {pred_name} ({confidence:.2f})",
                              fontsize=7, color="green" if artist == pred_name else "red")
            ax_orig.axis("off")

            # Plot overlay
            ax_cam = axes[row, col * 2 + 1]
            ax_cam.imshow(overlay)
            ax_cam.set_title("Grad-CAM", fontsize=7)
            ax_cam.axis("off")

    plt.tight_layout()
    out_path = f"gradcam_{model_name}.png"
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved to {out_path}")


# ── Run Grad-CAM for all CNN-based models ─────────────────────────────────────
for name, model in models.items():
    if name == "DINOv2":
        print(f"Skipping Grad-CAM for {name} (ViT — use attention rollout instead)")
        continue
    print(f"\nGenerating Grad-CAM grid for {name}...")
    visualise_gradcam_grid(
        model, name, class_names, DATA_DIR,
        n_artists=6, n_per_artist=3
    )


## Tip: reading Grad-CAM outputs for the report

**Good (model is learning style):** activation spreads across the canvas,
highlighting brushstroke texture, paint application, and edge quality.

**Bad (subject bias not fixed):** activation concentrates tightly on the
central subject (a boat, a tree, a face). The model is classifying the
*subject* rather than the *style* — it would fail on a different composition
by the same artist.

If you see the bad pattern predominantly, it means CoarseDropout augmentation
should be increased (higher `p` or larger `max_holes`).
